In [1]:
%pip install -q -U "crewai[google-genai]" ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.1/234.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.4/724.4 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25

In [2]:
from getpass import getpass
import os

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

print("Gemini API key configured.")

Enter your Gemini API key: ··········
Gemini API key configured.


In [3]:
from crewai import LLM

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.2
)

print(llm.call("Reply with exactly: W9D3 Gemini test successful"))

W9D3 Gemini test successful


In [4]:
from crewai import Agent, Task, Crew, Process

researcher = Agent(
    role="Research Analyst",
    goal="Research benefits, risks, and applications of Generative AI in Education",
    backstory="You are a careful researcher who gathers relevant evidence and identifies limitations.",
    llm=llm,
    verbose=False
)

writer = Agent(
    role="Report Writer",
    goal="Turn research findings into a clear and balanced report",
    backstory="You write structured reports that students and educators can understand.",
    llm=llm,
    verbose=False
)

reviewer = Agent(
    role="Quality Reviewer",
    goal="Check accuracy, balance, clarity, and completeness",
    backstory="You critically review reports and recommend practical improvements.",
    llm=llm,
    verbose=False
)

print("Researcher, Writer, and Reviewer created.")

Researcher, Writer, and Reviewer created.


In [5]:
research_task = Task(
    description=(
        "Research Generative AI in Education. Identify key benefits, risks, "
        "use cases, and challenges. Topic: {topic}"
    ),
    expected_output="Structured research notes covering benefits, risks, use cases, and challenges.",
    agent=researcher
)

writing_task = Task(
    description=(
        "Write a balanced report on {topic} using the research notes. "
        "Include an introduction, benefits, risks, examples, and conclusion."
    ),
    expected_output="A readable report with clear headings and practical examples.",
    agent=writer
)

review_task = Task(
    description=(
        "Review the report for clarity, balance, unsupported claims, and completeness. "
        "Provide an improved final version."
    ),
    expected_output="A reviewed final report with useful recommendations.",
    agent=reviewer
)

print("Three tasks created.")

Three tasks created.


In [6]:
crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[research_task, writing_task, review_task],
    process=Process.sequential,
    verbose=False
)

baseline_result = await crew.kickoff_async(
    inputs={"topic": "Generative AI in Education"}
)

print(baseline_result)

# Quality Reviewer's Evaluation & Improved Report

## Quality Review & Recommendations
* **Accuracy & Completeness:** The original report accurately reflects the source research notes. However, to make the report fully complete as an authoritative briefing document, I recommend explicitly integrating the **Research Limitations and Gaps** (longitudinal data deficits and geographic/contextual bias) into the conclusion or as a standalone final subsection. Decision-makers must understand the current limits of the empirical evidence.
* **Balance & Clarity:** The tone is appropriately objective, analytical, and well-structured. Subheadings and bullet points improve scan-ability. To enhance clarity, I have refined the transitional phrasing in the conclusion to emphasize actionable next steps rather than general observations.
* **Unsupported Claims:** All points are well-sourced from the foundational research notes; no extraneous speculation has been introduced.

---

# Improved Final Report



In [7]:
from ddgs import DDGS

search_results = DDGS().text(
    "Generative AI in education benefits risks",
    max_results=5
)

for item in search_results:
    print("Title:", item.get("title", ""))
    print("URL:", item.get("href", ""))
    print("Summary:", item.get("body", ""))
    print("-" * 60)

Title: Generative AI in education: Benefits, risks, and use cases
URL: https://geniusee.com/single-blog/generative-ai-education-benefits-risks-use-cases
Summary: Nov 19, 2024 · Generative AI can change education by making learning more personal and efficient. However, handling risks like data privacy, bias, and becoming too dependent on technology is essential.
------------------------------------------------------------
Title: Generative AI in Education: 7 Powerful Benefits, Real Risks ...
URL: https://aiexploretools.com/generative-ai-in-education/
Summary: Jan 30, 2026 · This article will discuss generative AI in education and its benefits, risks, and examples of how it is currently being used in classrooms and highlight the need for responsible and ethical use of generative AI in educational institutions.
------------------------------------------------------------
Title: Generation AI: Navigating the opportunities and risks of ...
URL: https://www.unesco.org/en/articles/generation-

In [8]:
from crewai.tools import tool
from ddgs import DDGS

@tool("Web Search")
def web_search(query: str) -> str:
    """Search the web for relevant information and return titles, URLs, and summaries."""
    results = DDGS().text(query, max_results=5)

    if not results:
        return "No web search results found."

    return "\n\n".join(
        f"Title: {r.get('title', '')}\n"
        f"URL: {r.get('href', '')}\n"
        f"Summary: {r.get('body', '')}"
        for r in results
    )

print("Web Search tool created.")

Web Search tool created.


In [9]:
researcher.tools = [web_search]

web_research_task = Task(
    description=(
        "Research {topic} using the Web Search tool. "
        "Include benefits, risks, current examples, and source URLs. "
        "Use the search results as evidence."
    ),
    expected_output="Research findings supported by web search results and source URLs.",
    agent=researcher
)

web_writing_task = Task(
    description=(
        "Write a balanced report on {topic} using the research findings. "
        "Include benefits, risks, examples, and source URLs."
    ),
    expected_output="A structured report with source URLs and practical examples.",
    agent=writer
)

web_review_task = Task(
    description=(
        "Review the report for clarity, balance, evidence, and completeness. "
        "Preserve source URLs and flag unsupported claims."
    ),
    expected_output="An improved, reviewed final report.",
    agent=reviewer
)

web_crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[web_research_task, web_writing_task, web_review_task],
    process=Process.sequential,
    verbose=False
)

print("Web-enabled research crew created.")

Web-enabled research crew created.


In [10]:
web_result = await web_crew.kickoff_async(
    inputs={"topic": "Generative AI in Education"}
)

print(web_result)

# Research Report: Generative AI in Education

## Introduction
Generative Artificial Intelligence (AI) has rapidly transitioned from a theoretical concept to an active, disruptive force across various sectors, with education being one of the most profoundly impacted. From automated lesson planning to hyper-personalized tutoring, generative AI tools are reshaping how students learn and how educators teach. This research report examines the core applications, benefits, potential risks, and current examples of generative AI in educational settings.

---

## 1. Applications of Generative AI in Education
Generative AI tools (such as large language models, text-to-image generators, and specialized educational software) are being applied across multiple facets of the learning ecosystem:

* **Personalized Tutoring and Learning Assistants:** AI systems can adapt to individual student learning paces, identifying knowledge gaps and providing customized explanations, practice problems, and feedbac

In [11]:
print("BASELINE REPORT")
print(str(baseline_result)[:3000])

print("\n" + "=" * 60)
print("WEB-ENABLED REPORT")
print(str(web_result)[:3000])

BASELINE REPORT
# Quality Reviewer's Evaluation & Improved Report

## Quality Review & Recommendations
* **Accuracy & Completeness:** The original report accurately reflects the source research notes. However, to make the report fully complete as an authoritative briefing document, I recommend explicitly integrating the **Research Limitations and Gaps** (longitudinal data deficits and geographic/contextual bias) into the conclusion or as a standalone final subsection. Decision-makers must understand the current limits of the empirical evidence.
* **Balance & Clarity:** The tone is appropriately objective, analytical, and well-structured. Subheadings and bullet points improve scan-ability. To enhance clarity, I have refined the transitional phrasing in the conclusion to emphasize actionable next steps rather than general observations.
* **Unsupported Claims:** All points are well-sourced from the foundational research notes; no extraneous speculation has been introduced.

---

# Improve

In [12]:
# W9D3: Research Crew — Results and Improvement Documentation

improvement_report = """
TASK: Building a Research Crew — Multi-Agent Pipeline

TOPIC: Generative AI in Education

AGENTS:
1. Researcher — gathers research findings, benefits, risks, and examples.
2. Writer — creates a clear and structured research report.
3. Reviewer — checks clarity, balance, completeness, and unsupported claims.

BASELINE PIPELINE:
The three agents worked sequentially to research, write, and review
a report on Generative AI in Education.

WEB-ENABLED PIPELINE:
A Web Search tool was added to the Researcher agent to retrieve
relevant search results, summaries, and source URLs.

IMPROVEMENTS:
1. Web search provides additional information beyond the model's
   existing knowledge.
2. Source URLs make it easier to check the information.
3. The Writer organizes the research into a readable report.
4. The Reviewer checks the report for clarity, balance, and completeness.
5. The multi-agent workflow separates research, writing, and review.

RESULT COMPARISON:
The baseline report uses the original research workflow.
The web-enabled report can include search-based evidence and source URLs.
Actual improvements depend on the search results and final report quality.

CONCLUSION:
The CrewAI multi-agent pipeline demonstrates how specialized agents
can collaborate on research and report generation. Integrating web search
helps provide additional evidence and makes sources easier to verify.
"""

print(improvement_report)


TASK: Building a Research Crew — Multi-Agent Pipeline

TOPIC: Generative AI in Education

AGENTS:
1. Researcher — gathers research findings, benefits, risks, and examples.
2. Writer — creates a clear and structured research report.
3. Reviewer — checks clarity, balance, completeness, and unsupported claims.

BASELINE PIPELINE:
The three agents worked sequentially to research, write, and review
a report on Generative AI in Education.

WEB-ENABLED PIPELINE:
A Web Search tool was added to the Researcher agent to retrieve
relevant search results, summaries, and source URLs.

IMPROVEMENTS:
1. Web search provides additional information beyond the model's
   existing knowledge.
2. Source URLs make it easier to check the information.
3. The Writer organizes the research into a readable report.
4. The Reviewer checks the report for clarity, balance, and completeness.
5. The multi-agent workflow separates research, writing, and review.

RESULT COMPARISON:
The baseline report uses the original r